![](/Workspace/Users/sunnygupta2508@gmail.com/Databricks-Certified-Data-Engineer-Pro/Includes/images/orders.png)

In [0]:
%run ../Includes/Copy-Datasets

In [0]:
df_bronze = spark.table("bronze")
display(df_bronze.head(5))

In [0]:
%sql
select schema_of_json(value::string)
from bronze
where topic = 'orders'
limit 1
--books: ARRAY<STRUCT<book_id: STRING, quantity: BIGINT, subtotal: BIGINT>>, customer_id: STRING, order_id: STRING, order_timestamp: STRING, quantity: BIGINT, total: BIGINT

In [0]:
from pyspark.sql import functions as F
json_schema = json_schema = "order_id STRING, order_timestamp Timestamp, customer_id STRING, quantity BIGINT, total BIGINT, books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>"
query = (spark.readStream.table("bronze")
                .filter("topic = 'orders'")
                .select(F.from_json(F.col("value").cast("string"), json_schema).alias("v"))
                .select("v.*")
            .writeStream
                .option("checkpointLocation", f"{bookstore.checkpoint_path}/Orders_silver")
                .trigger(availableNow=True)
                .table("orders_silver")
         )
query.awaitTermination()

In [0]:
df_orders_silver = spark.table("orders_silver")
display(df_orders_silver.head(5))


In [0]:
display(df_orders_silver.count())